# Phase 1: Voice Screener
This notebook trains and evaluates ML classifiers to detect Parkinson's Disease from voice features.

**Dataset:** `VikasUkani.data` (195 recordings, 22 acoustic features)
- `status = 1` → Parkinson's (147 samples)
- `status = 0` → Healthy (48 samples)

**v2 Production Model:** `GradientBoostingClassifier` tuned with `GridSearchCV` over a 5-fold Stratified K-Fold.
This replaced the original `RandomForestClassifier` and improved robustness on the imbalanced dataset.

**Saved to:** `backend/models/voice_model.pkl` and `backend/models/voice_scaler.pkl`

In [1]:
# 1. Imports
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# 2. Load Dataset (path relative to notebooks/ folder)
df = pd.read_csv('../data/VikasUkani.data')
print('Dataset shape:', df.shape)
print('Class distribution:')
print(df['status'].value_counts())

# 3. Separate Features (X) and Target (y)
X = df.drop(['name', 'status'], axis=1)  # Drop name and label
y = df['status']                          # 1 = Parkinson's, 0 = Healthy

# 4. Stratified train/test split (preserves class ratio in both splits)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 5. Scale the Data — use .values to strip column names for backend compatibility
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train.values)
X_test_scaled  = scaler.transform(X_test.values)

print(f'\nTrain size: {len(X_train)} | Test size: {len(X_test)}')

Dataset shape: (195, 24)
Class distribution:
status
1    147
0     48
Name: count, dtype: int64

Train size: 156 | Test size: 39


In [2]:
# ==========================================
# BASELINE MODELS (for comparison)
# ==========================================

# --- Logistic Regression ---
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train_scaled, y_train)
log_preds = log_model.predict(X_test_scaled)
print('--- 1. LOGISTIC REGRESSION ---')
print(f'Accuracy: {accuracy_score(y_test, log_preds) * 100:.2f}%')
print('Confusion Matrix:\n', confusion_matrix(y_test, log_preds), '\n')

# --- Support Vector Machine ---
svm_model = SVC(kernel='linear')
svm_model.fit(X_train_scaled, y_train)
svm_preds = svm_model.predict(X_test_scaled)
print('--- 2. SUPPORT VECTOR MACHINE ---')
print(f'Accuracy: {accuracy_score(y_test, svm_preds) * 100:.2f}%')
print('Confusion Matrix:\n', confusion_matrix(y_test, svm_preds), '\n')

--- 1. LOGISTIC REGRESSION ---
Accuracy: 92.31%
Confusion Matrix:
 [[ 8  2]
 [ 1 28]] 

--- 2. SUPPORT VECTOR MACHINE ---
Accuracy: 94.87%
Confusion Matrix:
 [[ 8  2]
 [ 0 29]] 



In [3]:
# ==========================================
# PRODUCTION MODEL: GradientBoosting + GridSearchCV
# ==========================================
# GradientBoosting trains trees sequentially — each new tree corrects the
# errors of the previous one. Consistently outperforms RandomForest on
# small, imbalanced tabular datasets like this one.
#
# GridSearchCV exhaustively tests all parameter combinations using
# 5-fold Stratified K-Fold cross-validation (preserves class ratio per fold).

param_grid = {
    'n_estimators':     [200, 400],
    'max_depth':        [3, 4, 5],
    'learning_rate':    [0.05, 0.1],
    'subsample':        [0.8, 1.0],
    'min_samples_leaf': [1, 3],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
gs = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1,
)

print('Running GridSearchCV — testing 48 parameter combinations...')
gs.fit(X_train_scaled, y_train)

best_model = gs.best_estimator_
print(f'\nBest params: {gs.best_params_}')
print(f'CV accuracy: {gs.best_score_ * 100:.2f}%')

# Evaluate on held-out test set
gb_preds = best_model.predict(X_test_scaled)
print('\n--- 3. GRADIENT BOOSTING (Production Model) ---')
print(f'Test Accuracy: {accuracy_score(y_test, gb_preds) * 100:.2f}%')
print('Confusion Matrix:\n', confusion_matrix(y_test, gb_preds))
print('\nClassification Report:')
print(classification_report(y_test, gb_preds, target_names=['Healthy', "Parkinson's"]))

Running GridSearchCV — testing 48 parameter combinations...
Fitting 5 folds for each of 48 candidates, totalling 240 fits

Best params: {'learning_rate': 0.1, 'max_depth': 3, 'min_samples_leaf': 1, 'n_estimators': 400, 'subsample': 0.8}
CV accuracy: 93.61%

--- 3. GRADIENT BOOSTING (Production Model) ---
Test Accuracy: 94.87%
Confusion Matrix:
 [[ 9  1]
 [ 1 28]]

Classification Report:
              precision    recall  f1-score   support

     Healthy       0.90      0.90      0.90        10
 Parkinson's       0.97      0.97      0.97        29

    accuracy                           0.95        39
   macro avg       0.93      0.93      0.93        39
weighted avg       0.95      0.95      0.95        39



In [4]:
# ==========================================
# SAVE MODEL + SCALER
# ==========================================
# The scaler MUST be saved alongside the model.
# At inference time, backend/utils.py extracts raw features — the scaler
# transforms them into the same distribution as the training data.

joblib.dump(best_model, '../backend/models/voice_model.pkl')
joblib.dump(scaler,     '../backend/models/voice_scaler.pkl')
print('[saved] voice_model.pkl  -> GradientBoostingClassifier')
print('[saved] voice_scaler.pkl -> StandardScaler')
print(f'\nFinal Test Accuracy: {accuracy_score(y_test, gb_preds) * 100:.2f}%')

[saved] voice_model.pkl  -> GradientBoostingClassifier
[saved] voice_scaler.pkl -> StandardScaler

Final Test Accuracy: 94.87%
